In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 1. Cấu hình đường dẫn
# Thay đổi đường dẫn này trỏ đến file 30 phút đã gộp 5 trạm của bạn
INPUT_PATH = Path("TanHung_30min_cleaned_4stations_6years.csv") 

print(f"--- ĐANG ĐỌC DỮ LIỆU TỪ: {INPUT_PATH.name} ---")

# 2. Đọc dữ liệu
df = pd.read_csv(INPUT_PATH)

# 3. Ép kiểu dữ liệu thời gian và sắp xếp
df['TIME'] = pd.to_datetime(df['TIME'])
df = df.sort_values(['MaDiemDo', 'TIME']).reset_index(drop=True)

# 4. Sao lưu cột P gốc để đối chiếu sau khi xử lý
df['P_kW_raw'] = df['P_kW']
df['P_clean'] = df['P_kW'] # Cột này sẽ dùng để thực hiện các bước làm sạch
df['fill_method'] = 'original' # Ghi chú phương pháp xử lý cho từng dòng

# 5. Khai báo các cột thông số để tính toán vật lý
num_cols = ['UA', 'UB', 'UC', 'IA', 'IB', 'IC', 'CosF', 'P_kW']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f" Đã tải: {len(df):,} records")
print(f"Khoảng thời gian: {df['TIME'].min()} --> {df['TIME'].max()}")
print(f"Các trạm hiện có: {df['MaDiemDo'].unique().tolist()}")

--- ĐANG ĐỌC DỮ LIỆU TỪ: TanHung_30min_cleaned_4stations_6years.csv ---
 Đã tải: 413,376 records
Khoảng thời gian: 2020-02-09 00:00:00 --> 2025-12-31 23:30:00
Các trạm hiện có: [4295, 4447, 4587, 8667]


In [2]:
# =============================================================================
# BƯỚC 1: XỬ LÝ ZERO NGHI NGỜ (SUSPICIOUS ZEROS)
# =============================================================================

print("--- ĐANG THỰC HIỆN BƯỚC 1: SỬA LỖI SENSOR BÁO 0 KHI ĐỦ 3U VÀ 3I ---")

# 1. KIỂM TRA ĐIỀU KIỆN ĐỦ: Phải có đủ 3 pha U và 3 pha I + CosF
# Đây là "chốt chặn" quan trọng nhất theo yêu cầu của bạn
df['has_full_signals'] = df[['UA', 'UB', 'UC', 'IA', 'IB', 'IC', 'CosF']].notna().all(axis=1)

# 2. CÔNG CỤ NỀN TẢNG: Tính toán P_phys và I_max
# Chỉ tính P_phys nếu có đầy đủ các tín hiệu trên
df['P_phys'] = np.where(
    df['has_full_signals'],
    (df['UA'] * df['IA'] + df['UB'] * df['IB'] + df['UC'] * df['IC']) * df['CosF'].abs() / 1_000_000,
    np.nan
)
df['I_max'] = df[['IA', 'IB', 'IC']].abs().max(axis=1)

# 3. AUDIT: In 5 dòng có P=0 nhưng đầy đủ 3U, 3I (Kiểm chứng lỗi Sensor)
mask_p0_audit = (df['P_clean'] == 0) & (df['has_full_signals']) & (df['I_max'] > 0.5)

if mask_p0_audit.any():
    print(f"\n[AUDIT] Tìm thấy {mask_p0_audit.sum():,}. Dưới đây là 5 mẫu P=0 nhưng có đủ 3U, 3I và CosF:")
    # In đầy đủ các pha để bạn kiểm tra
    audit_cols = ['TIME', 'TenDiemDo', 'UA', 'UB', 'UC', 'IA', 'IB', 'IC', 'CosF', 'P_kW_raw', 'P_phys']
    print(df[mask_p0_audit][audit_cols].head(5).to_string(index=False))

# 4. THỰC HIỆN XỬ LÝ
I_THRESHOLD = 0.5
P_PHYS_THRESHOLD = 1.0

# Điều kiện sửa: P=0 VÀ (Phải đủ 3U, 3I) VÀ (Có tải thực)
mask_suspicious_zero = (
    (df['P_clean'] == 0) & 
    (df['has_full_signals']) & 
    ((df['I_max'] > I_THRESHOLD) | (df['P_phys'] > P_PHYS_THRESHOLD))
)

zeros_before = (df['P_kW_raw'] == 0).sum()
count_zero_fixed = mask_suspicious_zero.sum()
df.loc[mask_suspicious_zero, 'fill_method'] = 'zero_replace_P_phys'
df.loc[mask_suspicious_zero, 'P_clean'] = df.loc[mask_suspicious_zero, 'P_phys']

# 5. THỐNG KÊ CHI TIẾT
print(f"\n[THỐNG KÊ BƯỚC 1]:")
print(f"- Tổng số mẫu P = 0 ban đầu: {zeros_before:,}")
print(f"- Số mẫu có đủ 3U, 3I nhưng P=0: {mask_p0_audit.sum():,}")
print(f"- Số mẫu 'Zero nghi ngờ' thỏa mãn điều kiện và ĐÃ SỬA: {count_zero_fixed:,}")
print(f"- Số mẫu P = 0 còn lại (Không đủ tín hiệu hoặc không tải): {(df['P_clean'] == 0).sum():,}")

if count_zero_fixed > 0:
    print("\nPhân bố sửa lỗi theo trạm:")
    print(df[df['fill_method'] == 'zero_replace_P_phys'].groupby('TenDiemDo').size())

print("\n HOÀN THÀNH BƯỚC 1: Dữ liệu Zero đã được khôi phục dựa trên 6 thông số pha.")

--- ĐANG THỰC HIỆN BƯỚC 1: SỬA LỖI SENSOR BÁO 0 KHI ĐỦ 3U VÀ 3I ---

[AUDIT] Tìm thấy 363. Dưới đây là 5 mẫu P=0 nhưng có đủ 3U, 3I và CosF:
               TIME    TenDiemDo       UA       UB       UC     IA    IB   IC  CosF  P_kW_raw   P_phys
2020-10-17 06:00:00 474 Tân Hưng 13240.58 13243.14 13288.17 115.46 96.17 89.0   1.0       0.0 3.984997
2020-10-31 06:00:00 474 Tân Hưng 13142.61 13151.05 13193.51   2.75  2.08  0.0   1.0       0.0 0.063496
2021-04-12 15:30:00 474 Tân Hưng 12959.70 12964.30 12975.04   0.00 23.01  0.0   1.0       0.0 0.298309
2021-05-05 15:00:00 474 Tân Hưng 13070.21 13069.70 13119.84  27.66  0.00  0.0   1.0       0.0 0.361522
2021-05-14 15:00:00 474 Tân Hưng 13079.67 13061.77 13068.67   0.00 24.13  0.0   1.0       0.0 0.315181

[THỐNG KÊ BƯỚC 1]:
- Tổng số mẫu P = 0 ban đầu: 34,802
- Số mẫu có đủ 3U, 3I nhưng P=0: 363
- Số mẫu 'Zero nghi ngờ' thỏa mãn điều kiện và ĐÃ SỬA: 363
- Số mẫu P = 0 còn lại (Không đủ tín hiệu hoặc không tải): 34,439

Phân bố sửa lỗi theo t

In [3]:
# =============================================================================
# BƯỚC 2: ĐÁNH GIÁ SAI SỐ VẬT LÝ THEO TỶ LỆ % (CHỈ XÉT P > 0)
# =============================================================================

print("--- ĐANG THỰC HIỆN BƯỚC 2: ĐÁNH GIÁ SAI SỐ TRÊN TẢI DƯƠNG ---")

# 1. Tính toán sai số tuyệt đối
df['P_diff'] = (df['P_clean'] - df['P_phys']).abs()
df['P_mag_diff'] = (df['P_clean'].abs() - df['P_phys']).abs()

# 2. Tính toán sai số phần trăm (%) - Chỉ tính khi P_clean > 0 và P_phys > 0.1
# Công thức: $P_{error\%} = \frac{|P_{clean} - P_{phys}|}{P_{phys}} \times 100$
df['P_error_pct'] = np.where(
    (df['P_clean'] > 0) & (df['P_phys'] > 0.1), 
    (df['P_diff'] / df['P_phys']) * 100, 
    np.nan # Trả về NaN cho các trường hợp P âm hoặc không tải để không tính vào thống kê
)

# 3. Cờ mâu thuẫn dấu (Vẫn giữ để Bước 3 xử lý đảo dây)
df['flag_sign_conflict'] = (df['P_clean'] < 0) & (df['P_phys'] > 0) & (df['P_mag_diff'] < 0.5)

# 4. THỐNG KÊ CHI TIẾT (Chỉ lọc các mẫu P dương)
valid_mask = df['P_error_pct'].notna()

print(f"\n[THỐNG KÊ BƯỚC 2]:")
print(f"- Số mẫu tải dương (>0.1kW) để đánh giá độ chính xác: {valid_mask.sum():,}")
print(f"- Sai số % trung bình trên tải dương: {df.loc[valid_mask, 'P_error_pct'].mean():.2f}%")

print("\nPhân bố sai số theo tỷ lệ % (Chỉ xét P > 0):")
bins = [0, 5, 10, np.inf]
labels = ['< 5% (Khớp tốt)', '5-10% (Chấp nhận)', '> 10% (Lệch nặng)']
error_dist = pd.cut(df.loc[valid_mask, 'P_error_pct'], bins=bins, labels=labels, include_lowest=True).value_counts().sort_index()

for label, count in error_dist.items():
    print(f"  + {label:<20}: {count:>8,} mẫu ({count/valid_mask.sum()*100:>6.2f}%)")

print(f"\n- Số mẫu mâu thuẫn dấu (P âm nhưng khớp vật lý): {df['flag_sign_conflict'].sum():,}")
print("\n HOÀN THÀNH BƯỚC 2")

--- ĐANG THỰC HIỆN BƯỚC 2: ĐÁNH GIÁ SAI SỐ TRÊN TẢI DƯƠNG ---

[THỐNG KÊ BƯỚC 2]:
- Số mẫu tải dương (>0.1kW) để đánh giá độ chính xác: 337,810
- Sai số % trung bình trên tải dương: 1.95%

Phân bố sai số theo tỷ lệ % (Chỉ xét P > 0):
  + < 5% (Khớp tốt)     :  320,244 mẫu ( 94.80%)
  + 5-10% (Chấp nhận)   :    8,253 mẫu (  2.44%)
  + > 10% (Lệch nặng)   :    9,313 mẫu (  2.76%)

- Số mẫu mâu thuẫn dấu (P âm nhưng khớp vật lý): 28,707

 HOÀN THÀNH BƯỚC 2


In [4]:
# =============================================================================
# BƯỚC 3: XỬ LÝ P ÂM (SOLAR VS LỖI ĐẤU NGƯỢC)
# =============================================================================

print("--- ĐANG THỰC HIỆN BƯỚC 3: PHÂN LOẠI VÀ XỬ LÝ P ÂM ---")

SOLAR_START, SOLAR_END = 7, 16
df['hour'] = df['TIME'].dt.hour
df['is_solar_time'] = df['hour'].between(SOLAR_START, SOLAR_END)

neg_before = (df['P_clean'] < 0).sum()

# 1. TRONG GIỜ SOLAR: Tất cả âm đưa về 0
mask_solar_zero = df['is_solar_time'] & (df['P_clean'] < 0)
df.loc[mask_solar_zero, 'P_clean'] = 0.0
df.loc[mask_solar_zero, 'fill_method'] = 'solar_neg_to_zero'

# 2. NGOÀI GIỜ SOLAR: Nếu khớp vật lý -> Đảo dấu
mask_night_flip = (~df['is_solar_time']) & df['flag_sign_conflict']
df.loc[mask_night_flip, 'P_clean'] = df.loc[mask_night_flip, 'P_clean'].abs()
df.loc[mask_night_flip, 'fill_method'] = 'night_sign_flip'

# 3. NGOÀI GIỜ SOLAR: Còn lại (Nhiễu) -> Về 0
mask_night_zero = (~df['is_solar_time']) & (df['P_clean'] < 0) & (~df['flag_sign_conflict'])
df.loc[mask_night_zero, 'P_clean'] = 0.0
df.loc[mask_night_zero, 'fill_method'] = 'night_neg_to_zero'

# 4. THỐNG KÊ CHI TIẾT
print(f"\n[THỐNG KÊ BƯỚC 3]:")
print(f"- Tổng số mẫu P âm ban đầu: {neg_before:,}")
print(f"- Đã xử lý Solar (7h-16h -> 0kW): {mask_solar_zero.sum():,}")
print(f"- Đã xử lý Đảo dây (Ban đêm -> Đảo dấu): {mask_night_flip.sum():,}")
print(f"- Đã xử lý Nhiễu ban đêm (Ban đêm -> 0kW): {mask_night_zero.sum():,}")
print(f"- Số mẫu P âm còn sót lại: {(df['P_clean'] < 0).sum():,}")

if mask_night_flip.sum() > 0:
    print("\nPhân bố trạm bị ngược TI (Night Flip):")
    print(df[df['fill_method'] == 'night_sign_flip'].groupby('TenDiemDo').size())

print("\n HOÀN THÀNH BƯỚC 3")

--- ĐANG THỰC HIỆN BƯỚC 3: PHÂN LOẠI VÀ XỬ LÝ P ÂM ---

[THỐNG KÊ BƯỚC 3]:
- Tổng số mẫu P âm ban đầu: 29,580
- Đã xử lý Solar (7h-16h -> 0kW): 29,572
- Đã xử lý Đảo dây (Ban đêm -> Đảo dấu): 5
- Đã xử lý Nhiễu ban đêm (Ban đêm -> 0kW): 3
- Số mẫu P âm còn sót lại: 0

Phân bố trạm bị ngược TI (Night Flip):
TenDiemDo
474 Tân Hưng    1
476 Tân Hưng    4
dtype: int64

 HOÀN THÀNH BƯỚC 3


In [5]:
# =============================================================================
# BƯỚC 4: LẤP ĐẦY DỮ LIỆU NaN TỪ NGUỒN VẬT LÝ
# =============================================================================

print("--- ĐANG THỰC HIỆN BƯỚC 4: CỨU DỮ LIỆU NaN ---")

nan_before = df['P_clean'].isna().sum()

mask_missing = df['P_clean'].isna() & df['P_phys'].notna()
df.loc[mask_missing, 'P_clean'] = df.loc[mask_missing, 'P_phys']
df.loc[mask_missing, 'fill_method'] = 'missing_from_P_phys'

# THỐNG KÊ CHI TIẾT
print(f"\n[THỐNG KÊ BƯỚC 4]:")
print(f"- Số lượng NaN ban đầu: {nan_before:,}")
print(f"- Số lượng được cứu bằng P_phys: {mask_missing.sum():,}")
print(f"- Số lượng NaN còn sót lại: {df['P_clean'].isna().sum():,}")

if mask_missing.sum() > 0:
    print("\nTrạm có nhiều dữ liệu NaN được cứu nhất:")
    print(df[df['fill_method'] == 'missing_from_P_phys'].groupby('TenDiemDo').size().sort_values(ascending=False).head(3))

print("\n HOÀN THÀNH BƯỚC 4. Dữ liệu đã sẵn sàng cho xử lý thống kê (MAD).")

--- ĐANG THỰC HIỆN BƯỚC 4: CỨU DỮ LIỆU NaN ---

[THỐNG KÊ BƯỚC 4]:
- Số lượng NaN ban đầu: 4,818
- Số lượng được cứu bằng P_phys: 0
- Số lượng NaN còn sót lại: 4,818

 HOÀN THÀNH BƯỚC 4. Dữ liệu đã sẵn sàng cho xử lý thống kê (MAD).


In [6]:
# =============================================================================
# BƯỚC 5: XỬ LÝ VI PHẠM VẬT LÝ NẶNG (LỆCH > 10%) - BẢN FIX TRIỆT ĐỂ
# =============================================================================

print("--- ĐANG THỰC HIỆN BƯỚC 5: THAY THẾ GIÁ TRỊ SAI SỐ > 10% ---")

# 1. QUAN TRỌNG: Tính toán lại sai số % cho TẤT CẢ các dòng sau khi đã đảo dấu ở Bước 3
# Chúng ta tính trên giá trị tuyệt đối để bao quát cả những sai lệch âm/dương
df['P_error_pct'] = np.where(
    df['has_full_signals'] & (df['P_phys'] > 0.1),
    (df['P_clean'].abs() - df['P_phys']).abs() / df['P_phys'] * 100,
    0
)

# 2. Định nghĩa ngưỡng lệch
HARD_ERROR_PCT_THRESHOLD = 10.0

# 3. Xác định các dòng vi phạm
# Lấy tất cả: Lệch > 10% HOẶC những ông vẫn đang <= 0 mà P_phys lại lớn
# Sửa lại Mask để KHÔNG đụng vào những dòng Solar đã xử lý ở Bước 2/3
mask_hard_violation = (
    df['has_full_signals'] & 
    (df['P_phys'] > 0.1) & 
    (~df['fill_method'].str.contains('solar|night_neg', na=False)) & # NE' SOLAR RA
    ((df['P_error_pct'] > HARD_ERROR_PCT_THRESHOLD) | (df['P_clean'] <= 0))
)

count_violation = mask_hard_violation.sum()
total_compare = df['has_full_signals'].sum()

# 4. THỰC HIỆN THAY THẾ: Ép về giá trị vật lý
df.loc[mask_hard_violation, 'P_clean'] = df.loc[mask_hard_violation, 'P_phys']
df.loc[mask_hard_violation, 'fill_method'] = 'hard_phys_fix_>10pct'

# 5. Cập nhật lại sai số lần cuối để báo cáo
df['P_diff'] = (df['P_clean'] - df['P_phys']).abs()
df['P_error_pct'] = np.where(
    (df['P_clean'] > 0) & (df['P_phys'] > 0.1), 
    (df['P_diff'] / df['P_phys']) * 100, 
    0
)

# =============================================================================
# THỐNG KÊ KẾT QUẢ BƯỚC 5
# =============================================================================
print(f"\n[THỐNG KÊ BƯỚC 5]:")
print(f"- Số mẫu phát hiện lệch nặng (>10% hoặc Zero lỗi): {count_violation:,}")
print(f"  => Đã đồng bộ toàn bộ về giá trị P_phys.")

# Tái thống kê để xác nhận nhóm > 10% đã về 0
valid_now = (df['P_clean'] > 0) & (df['P_phys'] > 0.1)
final_dist = pd.cut(df.loc[valid_now, 'P_error_pct'], 
                    bins=[0, 5, 10, np.inf], 
                    labels=['< 5%', '5-10%', '> 10%'], 
                    include_lowest=True).value_counts().sort_index()

print("\nPhân bố sai số tải dương SAU KHI XỬ LÝ:")
for label, count in final_dist.items():
    print(f"  + {label:<10}: {count:>8,} mẫu")

# Kiểm tra lại xem còn "kẻ đào tẩu" nào không
rogue_check = df[(df['P_error_pct'] > 10) & (df['has_full_signals']) & (df['P_phys'] > 0.1)].shape[0]
if rogue_check == 0:
    print("\n TRẠNG THÁI: HOÀN HẢO. Nhóm > 10% đã sạch bóng.")
else:
    print(f"\n CẢNH BÁO: Vẫn còn {rogue_check} mẫu chưa xử lý.")

print("\n HOÀN THÀNH BƯỚC 5")

--- ĐANG THỰC HIỆN BƯỚC 5: THAY THẾ GIÁ TRỊ SAI SỐ > 10% ---

[THỐNG KÊ BƯỚC 5]:
- Số mẫu phát hiện lệch nặng (>10% hoặc Zero lỗi): 9,314
  => Đã đồng bộ toàn bộ về giá trị P_phys.

Phân bố sai số tải dương SAU KHI XỬ LÝ:
  + < 5%      :  329,562 mẫu
  + 5-10%     :    8,253 mẫu
  + > 10%     :        0 mẫu

 TRẠNG THÁI: HOÀN HẢO. Nhóm > 10% đã sạch bóng.

 HOÀN THÀNH BƯỚC 5


In [7]:
# =============================================================================
# BƯỚC 6: PHÁT HIỆN OUTLIER THỐNG KÊ (MAD & JUMP DETECTOR)
# =============================================================================

print("--- ĐANG THỰC HIỆN BƯỚC 6: PHÁT HIỆN OUTLIER THỐNG KÊ ---")

# Hàm phát hiện Outlier bằng Median Absolute Deviation (MAD) - Rất mạnh cho dữ liệu điện
def detect_outliers_mad(series, threshold=6):
    median = series.median()
    mad = (series - median).abs().median()
    if mad == 0: return pd.Series(False, index=series.index)
    # Z-score hiệu chỉnh cho MAD
    z_score = 0.6745 * (series - median) / mad
    return z_score.abs() > threshold

# 1. Phát hiện Outlier theo từng mốc giờ (Seasonality) của từng Trạm
df['is_outlier'] = False
for station in df['MaDiemDo'].unique():
    for hour in range(24):
        mask = (df['MaDiemDo'] == station) & (df['hour'] == hour)
        if mask.any():
            # Chỉ xét những điểm chưa bị gán method 'solar_neg_to_zero' để tránh xóa nhầm Solar
            target_mask = mask & (df['fill_method'] != 'solar_neg_to_zero')
            if target_mask.any():
                df.loc[target_mask, 'is_outlier'] = detect_outliers_mad(df.loc[target_mask, 'P_clean'])

# 2. Phát hiện Bước nhảy đột ngột (Jump Detector)
# Nếu P thay đổi quá 50% và lệch > 10kW chỉ trong 30 phút (lỗi truyền tin)
df['P_prev'] = df.groupby('MaDiemDo')['P_clean'].shift(1)
df['jump'] = (df['P_clean'] - df['P_prev']).abs()
mask_jump = (df['jump'] > 5) & (df['jump'] > df['P_prev'] * 0.5) & (df['P_clean'].notna())
df['is_outlier'] = df['is_outlier'] | mask_jump

# 3. Chuyển Outlier về NaN để chuẩn bị cho nội suy ở Bước 7
outlier_count = df['is_outlier'].sum()
df.loc[df['is_outlier'], 'P_clean'] = np.nan
df.loc[df['is_outlier'], 'fill_method'] = 'statistical_outlier'

# =============================================================================
# THỐNG KÊ CHI TIẾT BƯỚC 6
# =============================================================================
print(f"\n[THỐNG KÊ BƯỚC 6]:")
print(f"- Tổng số Outlier phát hiện: {outlier_count:,}")
print(f"  + Lỗi biến động MAD (theo khung giờ): {outlier_count - mask_jump.sum():,}")
print(f"  + Lỗi bước nhảy đột ngột (Jump): {mask_jump.sum():,}")

if outlier_count > 0:
    print("\nChi tiết Outlier theo trạm:")
    print(df[df['is_outlier']].groupby('TenDiemDo').size())

print("\n HOÀN THÀNH BƯỚC 6: Các điểm dị biệt đã được đưa về NaN.")

--- ĐANG THỰC HIỆN BƯỚC 6: PHÁT HIỆN OUTLIER THỐNG KÊ ---

[THỐNG KÊ BƯỚC 6]:
- Tổng số Outlier phát hiện: 2,453
  + Lỗi biến động MAD (theo khung giờ): 748
  + Lỗi bước nhảy đột ngột (Jump): 1,705

Chi tiết Outlier theo trạm:
TenDiemDo
474 Tân Hưng    522
476 Tân Hưng    439
478 Tân Hưng    587
480 Tân Hưng    905
dtype: int64

 HOÀN THÀNH BƯỚC 6: Các điểm dị biệt đã được đưa về NaN.


In [8]:
# =============================================================================
# BƯỚC 7: NỘI SUY ĐA TẦNG (LINEAR <= 1H -> 4 TUẦN -> MÙA -> NĂM -> FALLBACK)
# =============================================================================

print("--- ĐANG THỰC HIỆN BƯỚC 7: NỘI SUY ĐA TẦNG & ĐỐI SOÁT ---")

# 1. Chuẩn bị đặc trưng thời gian
df['month'] = df['TIME'].dt.month
df['hour'] = df['TIME'].dt.hour
df['day_of_week'] = df['TIME'].dt.dayofweek
# Mùa khô (Dry: Tháng 1-4), Mùa mưa (Rainy: Tháng 5-12)
df['season'] = np.where(df['month'].between(1, 4), 'Dry', 'Rainy')

# 2. Tính toán bảng tham chiếu (Chỉ dùng dữ liệu P_clean hiện có)
seasonal_ref = df.groupby(['MaDiemDo', 'season', 'day_of_week', 'hour'])['P_clean'].mean()
yearly_ref = df.groupby(['MaDiemDo', 'month', 'day_of_week', 'hour'])['P_clean'].mean()

# Ghi lại danh sách NaN ban đầu của Bước 7
mask_initial_nan = df['P_clean'].isna()
nan_total_start = mask_initial_nan.sum()

# --- PHẦN 1: LINEAR CHO KHOẢNG TRỐNG CỰC NGẮN (<= 1 GIỜ / 2 MẪU) ---
df['P_clean'] = df.groupby('MaDiemDo')['P_clean'].transform(lambda x: x.interpolate(method='linear', limit=2))
mask_filled_linear_short = mask_initial_nan & df['P_clean'].notna()
df.loc[mask_filled_linear_short, 'fill_method'] = 'linear_short_gap'

# --- PHẦN 2: BỘ NÃO NỘI SUY ĐA TẦNG (HISTORICAL -> SEASONAL -> YEARLY) ---
def fill_missing_advanced(group):
    p_values = group['P_clean'].values.copy()
    nan_indices = np.where(pd.isna(p_values))[0]
    st_id = group.name # Lấy mã trạm
    
    WEEK_SLOTS = 336 # 7 ngày * 48 mẫu (30p)
    offsets = [WEEK_SLOTS, WEEK_SLOTS*2, WEEK_SLOTS*3, WEEK_SLOTS*4]
    
    for idx in nan_indices:
        original_idx = group.index[idx]
        
        # A. Ưu tiên 4 tuần gần nhất (Cùng Thứ, cùng Giờ)
        hist_vals = []
        for opt in offsets:
            prev_idx = idx - opt
            if prev_idx >= 0 and not pd.isna(p_values[prev_idx]):
                hist_vals.append(p_values[prev_idx])
        
        if hist_vals:
            p_values[idx] = np.mean(hist_vals)
            df.at[original_idx, 'fill_method'] = f'hist_avg_{len(hist_vals)}w'
            continue
            
        # B. Thử theo MÙA (Cùng Mùa, cùng Thứ, cùng Giờ)
        row = group.iloc[idx]
        try:
            val_seasonal = seasonal_ref.loc[(st_id, row['season'], row['day_of_week'], row['hour'])]
            if not pd.isna(val_seasonal):
                p_values[idx] = val_seasonal
                df.at[original_idx, 'fill_method'] = 'seasonal_avg_fallback'
                continue
        except: pass

        # C. Thử theo NĂM (Cùng Tháng, cùng Thứ, cùng Giờ)
        try:
            val_yearly = yearly_ref.loc[(st_id, row['month'], row['day_of_week'], row['hour'])]
            if not pd.isna(val_yearly):
                p_values[idx] = val_yearly
                df.at[original_idx, 'fill_method'] = 'yearly_avg_fallback'
        except: pass

    return pd.Series(p_values, index=group.index)

df['P_clean'] = df.groupby('MaDiemDo', group_keys=False).apply(fill_missing_advanced)

# --- PHẦN 3: LINEAR FALLBACK (Nếu không tìm thấy bất kỳ dữ liệu lịch sử nào) ---
mask_still_nan = df['P_clean'].isna()
df['P_clean'] = df.groupby('MaDiemDo')['P_clean'].transform(lambda x: x.interpolate(method='linear'))
mask_filled_fallback = mask_still_nan & df['P_clean'].notna()
df.loc[mask_filled_fallback, 'fill_method'] = 'linear_fallback_final'

# --- THỐNG KÊ BƯỚC 7 ---
print(f"\n[THỐNG KÊ BƯỚC 7]:")
print(f"- Tổng số mẫu NaN cần cứu: {nan_total_start:,}")
methods_step7 = ['linear_short_gap', 'hist_avg', 'seasonal_avg', 'yearly_avg', 'linear_fallback']
for m in methods_step7:
    count = (mask_initial_nan & df['fill_method'].str.contains(m, na=False)).sum()
    print(f"  + {m:<25}: {count:>8,} mẫu ({count/nan_total_start*100:>6.2f}%)")

print(f"- Số mẫu vẫn còn NaN: {df['P_clean'].isna().sum():,}")
print("\n HOÀN THÀNH BƯỚC 7")

--- ĐANG THỰC HIỆN BƯỚC 7: NỘI SUY ĐA TẦNG & ĐỐI SOÁT ---

[THỐNG KÊ BƯỚC 7]:
- Tổng số mẫu NaN cần cứu: 7,271
  + linear_short_gap         :    6,449 mẫu ( 88.69%)
  + hist_avg                 :      822 mẫu ( 11.31%)
  + seasonal_avg             :        0 mẫu (  0.00%)
  + yearly_avg               :        0 mẫu (  0.00%)
  + linear_fallback          :        0 mẫu (  0.00%)
- Số mẫu vẫn còn NaN: 0

 HOÀN THÀNH BƯỚC 7


In [9]:
# =============================================================================
# BƯỚC 8: TỔNG KIỂM KÊ (FINAL AUDIT REPORT)
# =============================================================================

print("\n" + "="*80)
print(f"{'BÁO CÁO KIỂM TRA CHẤT LƯỢNG DỮ LIỆU CUỐI CÙNG':^80}")
print("="*80)

# --- 1. KIỂM TRA P ÂM ---
neg_count = (df['P_clean'] < 0).sum()
print(f"1. KIỂM TRA CÒN P ÂM KHÔNG?")
if neg_count == 0:
    print(f"    Tuyệt vời: Không còn giá trị âm nào.")
else:
    print(f"    Cảnh báo: Vẫn còn {neg_count:,} mẫu âm. Cần xem lại Bước 3.")

# --- 2. KIỂM TRA P THIẾU (NaN) ---
nan_count = df['P_clean'].isna().sum()
print(f"\n2. KIỂM TRA P THIẾU (NaN):")
if nan_count == 0:
    print(f"    Tuyệt vời: Dữ liệu đã kín 100% (Không còn NaN).")
else:
    print(f"    Cảnh báo: Vẫn còn {nan_count:,} mẫu thiếu. Cần xem lại Bước 7.")

# --- 3. KIỂM TRA DESCRIPTION P (THỐNG KÊ MÔ TẢ) ---
print(f"\n3. KIỂM TRA DESCRIPTION P_clean (kW):")
desc = df['P_clean'].describe()
print(desc.to_frame().T.to_string())

# --- 4. KIỂM TRA LỆCH VẬT LÝ ---
# Tính toán lại sai số cuối cùng để đảm bảo tính thời sự
df['final_error_pct'] = np.where(
    (df['P_clean'] > 0) & (df['P_phys'] > 0.1),
    (df['P_clean'] - df['P_phys']).abs() / df['P_phys'] * 100,
    np.nan
)
valid_mask = df['final_error_pct'].notna()

print(f"\n4. KIỂM TRA ĐỘ LỆCH VẬT LÝ (|P_clean - P_phys| / P_phys):")
print(f"   - Sai số phần trăm trung bình: {df.loc[valid_mask, 'final_error_pct'].mean():.2f}%")

bins = [0, 5, 10, np.inf]
labels = ['Dưới 5% (Khớp)', '5-10% (Chấp nhận)', 'Trên 10% (Lệch)']
dist = pd.cut(df.loc[valid_mask, 'final_error_pct'], bins=bins, labels=labels, include_lowest=True).value_counts().sort_index()

for label, count in dist.items():
    print(f"   + {label:<20}: {count:>8,} mẫu ({count/valid_mask.sum()*100:>6.2f}%)")

# --- PHÂN TÍCH RIÊNG NHÓM LỆCH TRÊN 10% ---
mismatch_10 = df[valid_mask & (df['final_error_pct'] > 10)]
if not mismatch_10.empty:
    print(f"\n    PHÂN TÍCH NHÓM LỆCH > 10% ({len(mismatch_10):,} mẫu):")
    print("   (Phần lớn là do dữ liệu nội suy không thể khớp 100% với U, I thực thời điểm đó)")
    print(mismatch_10['fill_method'].value_counts().to_string())

print("\n" + "="*80)
print(f"{' KẾT LUẬN: DỮ LIỆU ĐÃ SẴN SÀNG ĐỂ TRAINING MODEL KAN':^80}")
print("="*80)


                 BÁO CÁO KIỂM TRA CHẤT LƯỢNG DỮ LIỆU CUỐI CÙNG                  
1. KIỂM TRA CÒN P ÂM KHÔNG?
    Tuyệt vời: Không còn giá trị âm nào.

2. KIỂM TRA P THIẾU (NaN):
    Tuyệt vời: Dữ liệu đã kín 100% (Không còn NaN).

3. KIỂM TRA DESCRIPTION P_clean (kW):
            count      mean       std  min      25%      50%      75%        max
P_clean  413376.0  5.878141  4.210732  0.0  2.10938  6.12303  9.62401  20.093184

4. KIỂM TRA ĐỘ LỆCH VẬT LÝ (|P_clean - P_phys| / P_phys):
   - Sai số phần trăm trung bình: 2.34%
   + Dưới 5% (Khớp)      :  327,767 mẫu ( 97.01%)
   + 5-10% (Chấp nhận)   :    8,231 mẫu (  2.44%)
   + Trên 10% (Lệch)     :    1,882 mẫu (  0.56%)

    PHÂN TÍCH NHÓM LỆCH > 10% (1,882 mẫu):
   (Phần lớn là do dữ liệu nội suy không thể khớp 100% với U, I thực thời điểm đó)
fill_method
linear_short_gap    1597
hist_avg_4w          285

               KẾT LUẬN: DỮ LIỆU ĐÃ SẴN SÀNG ĐỂ TRAINING MODEL KAN              


In [10]:
# =============================================================================
# XUẤT DỮ LIỆU SẠCH RA FILE CSV (FINAL EXPORT)
# =============================================================================

# Định nghĩa tên file đầu ra
OUTPUT_FILE = "TanHung_Cleaned_For_KAN_6years.csv"

print(f"--- ĐANG TIẾN HÀNH XUẤT DỮ LIỆU RA FILE: {OUTPUT_FILE} ---")

# Chọn các cột cần thiết để giữ file nhẹ và tập trung cho Model KAN
# Bạn có thể giữ lại tất cả các cột nếu muốn đối soát sâu hơn
cols_to_export = [
    'TIME', 'MaDiemDo', 'TenDiemDo', 
    'UA', 'UB', 'UC', 'IA', 'IB', 'IC', 'CosF', 
    'P_kW_raw', 'P_clean', 'fill_method'
]

# Thực hiện lưu file
try:
    df[cols_to_export].to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f"\n✅ THÀNH CÔNG: Đã lưu {len(df):,} bản ghi vào file '{OUTPUT_FILE}'.")
    print(f"📍 Vị trí: Cùng thư mục với file chạy code của bạn.")
    
    # Tính dung lượng file xấp xỉ
    import os
    file_size = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
    print(f"📦 Dung lượng file: {file_size:.2f} MB")
    
except Exception as e:
    print(f"❌ LỖI khi lưu file: {e}")

print("\n" + "="*80)
print(f"{'DỮ LIỆU ĐÃ SẴN SÀNG CHO BƯỚC TRAINING MODEL KAN':^80}")
print("="*80)

--- ĐANG TIẾN HÀNH XUẤT DỮ LIỆU RA FILE: TanHung_Cleaned_For_KAN_6years.csv ---

✅ THÀNH CÔNG: Đã lưu 413,376 bản ghi vào file 'TanHung_Cleaned_For_KAN_6years.csv'.
📍 Vị trí: Cùng thư mục với file chạy code của bạn.
📦 Dung lượng file: 46.70 MB

                DỮ LIỆU ĐÃ SẴN SÀNG CHO BƯỚC TRAINING MODEL KAN                 
